# ML-03 — Framing the Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neilivanorencia/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

Week 1 showed that most "low CTR" flags are bad luck. This notebook turns the lane into a precise machine-learning task: what the output is, what it is judged against, how success is measured, what one row of data is, and where a learned model can genuinely help.

**Decisions locked here and kept unchanged through the capstone:**

- **K = 50.** The editor reviews 50 pages per cycle, so the main measure is Precision@50, with Precision@20 for a smaller review cycle, and the base rate printed beside both.
- **Capture@50.** The share of all lost clicks that the top 50 pages hold, so success is measured in clicks as well as pages.
- **The click shortfall drives the ranking.** Engagement only changes the recommended action, on pages that have analytics data.

**How to read it:** each section opens with the thinking, then a **Method** note explaining what the next code cell computes, then the output, then a **Finding** note on what it shows. Week 1's terms (expected clicks, missed clicks, p, real shortfall) still apply; the new ones are defined below.


## Key terms

| Term                                  | Technical meaning                                                                          | In plain words                                                                                  |
| ------------------------------------- | ------------------------------------------------------------------------------------------ | ----------------------------------------------------------------------------------------------- |
| **Window A**                          | The 30 days before the most recent 30 (`*_prev_30d` columns)                               | "Last month": the data the score is allowed to see                                              |
| **Window B**                          | The most recent 30 days (`*_last_30d` columns)                                             | "This month": held back to check whether the score was right                                    |
| **Client factor**                     | A client's actual clicks ÷ its expected clicks under the tier rates, with +1 added to both | How much better or worse a client's pages click than average                                    |
| **Client-aware expectation**          | Tier-based expected clicks × the client factor                                             | What the page should earn, given both its position _and_ how its client's pages usually perform |
| **Score**                             | Window A missed clicks, kept only when p < 0.01                                            | The number the queue is sorted by                                                               |
| **Persistent shortfall (the target)** | In window B: p < 0.05 _and_ clicks at most half of expected                                | The page fell clearly and substantially short again the following month                         |
| **Base rate**                         | The share of all pages where the target is true                                            | How often a random pick would be right                                                          |
| **Precision@K**                       | The share of the top K pages whose target is true                                          | Of the K pages the editor opens, how many really needed attention                               |
| **Lift**                              | Precision@K ÷ base rate                                                                    | How many times better than random picking                                                       |
| **Capture@K**                         | The top K pages' share of all window-B missed clicks                                       | How much of the total lost value the queue covers                                               |
| **Self-defined label**                | A label built from the same data and rule as the score                                     | Like a student grading their own homework: it always looks good                                 |
| **Client holdout**                    | Testing on clients whose pages were never used to build the method                         | A fair test: can it handle a client it has never seen?                                          |


## Setup

**Method, step by step:**

1. **Finds the data.** In Google Colab, the notebook starts on an empty machine, so it downloads a copy of the project (`git clone --depth 1` fetches only the latest version, which is faster). Run locally, it starts inside `work/notebooks/`, so it steps up one folder at a time until it reaches the project's top folder, where `data/raw/` lives.
2. **Loads the tools.** `pandas` handles tables and `scipy`'s `poisson` is the probability model used later to test whether a page's missing clicks could be bad luck.
3. **Loads the dataset** into a table called `df`, stopping with a clear message if the file is missing.
4. **Defines `show()` and `show_table()`**, two small helpers that display every result as a titled table in the same layout, so each cell's output is easy to scan.


In [1]:
import os
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import poisson

# Colab starts empty, so clone the project there; locally, walk up to the folder that holds data/raw.
if "google.colab" in sys.modules:
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/neilivanorencia/flyrank-ml-internship"], check=True)
    os.chdir("flyrank-ml-internship")
else:
    while not os.path.isdir("data/raw") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")

DATA = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(DATA), "Starter CSV not found: run this notebook from inside the project folder."
df = pd.read_csv(DATA)
pd.set_option("display.width", 160)
TIERS = ["top_3", "page_1", "striking", "page_3_5", "deep"]


NUMBER = r"^[-+]?[\d,]*\.?\d+(?:%|x)?$"


def show_table(title: str, frame: pd.DataFrame) -> None:
    """Display a DataFrame as a titled Markdown table.

    Markdown tables look the same in Jupyter, VS Code, Colab, and GitHub, and they avoid the extra
    index row and interactive-table scripts that a plain DataFrame display adds. Columns whose
    cells are all numbers are right-aligned.
    """
    body = frame.reset_index() if frame.index.name or isinstance(frame.index, pd.MultiIndex) else frame
    cells = body.astype(str).apply(lambda col: col.str.replace("|", "\\|", regex=False).str.replace("\n", " ", regex=False))
    align = ["---:" if cells[name].str.strip().str.match(NUMBER).all() else "---" for name in cells.columns]
    lines = ["| " + " | ".join(map(str, cells.columns)) + " |", "| " + " | ".join(align) + " |"]
    lines += ["| " + " | ".join(row) + " |" for row in cells.itertuples(index=False)]
    display(Markdown(f"**{title}**\n\n" + "\n".join(lines)))


def show(title: str, rows: dict[str, object]) -> None:
    """Display a titled two-column table of measures and their values."""
    show_table(title, pd.DataFrame({"Measure": list(rows), "Value": [str(value) for value in rows.values()]}))


show("Dataset loaded", {
    "pages (rows)": f"{len(df):,}",
    "columns": df.shape[1],
    "clients": df["client_id"].nunique(),
})

**Dataset loaded**

| Measure | Value |
| --- | ---: |
| pages (rows) | 30,000 |
| columns | 44 |
| clients | 32 |

### Shared definitions

**Method:** this cell defines four small functions used throughout this notebook, then builds the lane's table of pages.

- `tier_expected()` gives each page its expected clicks from its tier's pooled CTR (as in Week 1).
- `client_aware_expected()` adjusts that expectation for the page's client, so a client whose pages always click less is not mistaken for a site full of problems.
- `precision_at_k()` and `capture_at_k()` compute the two success measures defined above.

It then keeps pages with real position data and impressions in both windows, and computes expected clicks, p, and missed clicks separately for window A and window B.


In [ ]:
# Window A is the earlier 30 days and window B the most recent 30 days; they never overlap.
def tier_expected(frame: pd.DataFrame, clicks: str, impressions: str) -> pd.Series:
    """Return each page's expected clicks from the pooled CTR of its position tier.

    Pooling clicks and impressions per tier, rather than averaging page CTRs, keeps small pages
    from dragging the expectation around.
    """
    rate = frame.groupby("position_tier")[clicks].sum() / frame.groupby("position_tier")[impressions].sum()
    return frame[impressions] * frame["position_tier"].map(rate)


def client_aware_expected(frame: pd.DataFrame, clicks: str, impressions: str) -> pd.Series:
    """Return the tier expectation scaled by how each client's pages click relative to that norm.

    The +1 in the numerator and denominator keeps a client with very few clicks from receiving
    an extreme factor.
    """
    expected = tier_expected(frame, clicks, impressions)
    by_client = frame.assign(_expected=expected).groupby("client_id")
    factor = (by_client[clicks].sum() + 1) / (by_client["_expected"].sum() + 1)
    return expected * frame["client_id"].map(factor)


def precision_at_k(score: pd.Series, target: pd.Series, k: int) -> float:
    """Return the share of the top-k pages by score whose target is 1."""
    top = score.sort_values(ascending=False).index[:k]
    return float(target.loc[top].mean())


def capture_at_k(score: pd.Series, value: pd.Series, k: int) -> float:
    """Return the share of the total value (for example, missed clicks) that sits in the top-k pages."""
    top = score.sort_values(ascending=False).index[:k]
    return float(value.loc[top].sum() / value.sum())


# The lane frame: pages with real position data and impressions in both 30-day windows.
lane = df[(df["avg_position"] > 0) & (df["impressions_prev_30d"] > 0) & (df["impressions_last_30d"] > 0)].copy()
for tag, clicks, impressions in [("A", "clicks_prev_30d", "impressions_prev_30d"),
                                 ("B", "clicks_last_30d", "impressions_last_30d")]:
    lane[f"expected_{tag}"] = client_aware_expected(lane, clicks, impressions)
    lane[f"p_short_{tag}"] = poisson.cdf(lane[clicks], lane[f"expected_{tag}"])
    lane[f"missed_{tag}"] = (lane[f"expected_{tag}"] - lane[clicks]).clip(lower=0)
    lane[f"ratio_{tag}"] = lane[clicks] / lane[f"expected_{tag}"]
lane["expected_A_tier"] = tier_expected(lane, "clicks_prev_30d", "impressions_prev_30d")
lane["ctr_A"] = 100 * lane["clicks_prev_30d"] / lane["impressions_prev_30d"]

show("Lane frame built", {
    "pages with position data and impressions in both windows": f"{len(lane):,}",
    "window A": "the earlier 30 days (used to score pages)",
    "window B": "the most recent 30 days (used only to judge the score)",
})

**Lane frame built**

| Measure | Value |
| --- | --- |
| pages with position data and impressions in both windows | 25,214 |
| window A | the earlier 30 days (used to score pages) |
| window B | the most recent 30 days (used only to judge the score) |

## 1. My lane as an ML task (type)

**Ranking / scoring, built in two stages.**

1. **Expected clicks:** an estimate of what a page _should_ earn from its impressions, given its position, its client, and later its content type and intent. Statistically this is a regression problem, and it is where a learned model can earn its place.
2. **The score:** missed clicks from real shortfalls, meaning expected minus actual clicks, kept only when luck would produce the gap less than 1 time in 100. Sorting pages by this score gives the queue.

Why not the other task types:

- **Not classification.** A yes/no "opportunity" label would have to be invented, and a model trained on an invented label just learns the rule back. A yes/no outcome does appear in Section 2, but only as something _observed_ a month later to judge the score, never as something the score is fitted to.
- **Not clustering.** Clusters say which kinds of pages exist (that is Lane 3). They cannot say which page in a cluster to open first.

**Method:** this cell shows both stages on six real pages: the three highest-scoring pages, then three zero-click pages that a CTR-gap rule would flag. It explains why a page with _zero_ clicks can rank _below_ pages that earned clicks.


In [3]:
# The three top-scoring pages, then three zero-click pages a CTR-gap rule would flag.
lane["score_A"] = lane["missed_A"] * (lane["p_short_A"] < 0.01)
lane["tier_median_A"] = lane.groupby("position_tier")["ctr_A"].transform("median")
gap_flag = (lane["ctr_A"] < 0.5 * lane["tier_median_A"]) & (lane["impressions_prev_30d"] >= 100)

examples = pd.concat([
    lane.sort_values("score_A", ascending=False).head(3),
    lane[gap_flag & (lane["clicks_prev_30d"] == 0) & (lane["p_short_A"] > 0.05)]
    .sort_values("impressions_prev_30d", ascending=False).head(3),
])
example_table = pd.DataFrame({
    "Tier": examples["position_tier"],
    "Impressions (A)": examples["impressions_prev_30d"].map("{:,}".format),
    "Clicks (A)": examples["clicks_prev_30d"].map("{:,}".format),
    "Expected clicks": examples["expected_A"].map("{:,.1f}".format),
    "Missed clicks": examples["missed_A"].map("{:,.1f}".format),
    "p (luck)": examples["p_short_A"].map("{:.4f}".format),
    "Score": examples["score_A"].map("{:,.1f}".format),
}).set_index(examples["content_id"])
example_table.index.name = "Page (pseudonym)"
show_table("Top three pages by score, then three zero-click pages a CTR-gap rule would flag", example_table)

**Top three pages by score, then three zero-click pages a CTR-gap rule would flag**

| Page (pseudonym) | Tier | Impressions (A) | Clicks (A) | Expected clicks | Missed clicks | p (luck) | Score |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: |
| content_8c19996aa890 | top_3 | 161,284 | 389 | 821.5 | 432.5 | 0.0000 | 432.5 |
| content_5fe46e04994d | page_1 | 218,786 | 250 | 675.7 | 425.7 | 0.0000 | 425.7 |
| content_c8e9d6ab9013 | page_1 | 111,885 | 0 | 316.5 | 316.5 | 0.0000 | 316.5 |
| content_98b26cdb77a5 | page_1 | 3,220 | 0 | 2.8 | 2.8 | 0.0629 | 0.0 |
| content_d6e1bbb4a996 | page_1 | 2,755 | 0 | 2.4 | 2.4 | 0.0938 | 0.0 |
| content_8ba781dafa55 | page_1 | 2,431 | 0 | 2.6 | 2.6 | 0.0758 | 0.0 |

**Finding:** the three zero-click pages look alarming on a CTR chart, but each was expected to earn fewer than 3 clicks, so getting zero is unremarkable (luck explains it 6% to 9% of the time), and their score is zero. The top pages were expected to earn hundreds of clicks and earned far fewer, with p effectively zero. Ranking by missed clicks from real shortfalls puts those first.


## 2. Target or proxy

Three different things get called "the target" in this lane, and keeping them apart is the core of the framing.

|                                   | Built from                                    | Role                             |
| --------------------------------- | --------------------------------------------- | -------------------------------- |
| **Score**                         | Window A clicks and impressions only          | What the queue is sorted by      |
| **Observed target**               | Window B, which the score never sees          | What the queue is judged against |
| **Self-defined label** (the trap) | The same window and the same gap as the score | Makes any ranking look good      |

**The observed target, `persistent_shortfall`:** in the most recent 30 days, the page earned clearly fewer clicks than its client-aware expectation (p below 0.05) _and_ at most half of what was expected. Window B's clicks and impressions are Search Console measurements, so this is an outcome the data recorded, not a rule I wrote. The "at most half" condition stops very large pages from qualifying on a gap that is statistically clear but too small to matter.

**Why this is not circular:** the score only uses window A. If a window-A shortfall were just luck, it would usually not show up again in window B.

**Limitations:**

- **Position overlaps the target window.** Position is a 90-day average covering both windows, so a little information about window B leaks into the expectation. The full warehouse records position daily, which would remove this overlap.
- **Repeating is not the same as fixable.** A page can fall short every month because of Google's AI answers or rich results that no rewrite can change. The editor's judgement is still required.
- **The window-B client factor uses window-B data.** That is fine for _defining_ the outcome, but it is not known in advance, so it never enters the score.

**Method:** this cell builds the target and reports its base rate, then demonstrates the trap by judging the same CTR-gap ranking two ways: against a label made from its own gap, and against the observed window-B target.


In [4]:
# The target: a clear and substantial shortfall again in window B, which the score never sees.
lane["persistent_shortfall"] = ((lane["p_short_B"] < 0.05) & (lane["ratio_B"] <= 0.5)).astype(int)
target = lane["persistent_shortfall"]
base_rate = target.mean()
show("The observed target", {
    "pages with a persistent shortfall": f"{target.sum():,} of {len(lane):,}",
    "base rate (chance of a random pick being right)": f"{base_rate:.1%}",
})

# The trap: the same ranking judged against its own self-defined label, then against the real outcome.
gap_score = lane["tier_median_A"] - lane["ctr_A"]
self_label = gap_flag.astype(int)
show("CTR-gap ranking, judged two ways (Precision@50)", {
    "against its own self-defined label": f"{precision_at_k(gap_score, self_label, 50):.2f}",
    "against the observed window-B target": f"{precision_at_k(gap_score, target, 50):.2f}",
    "base rate, for comparison": f"{base_rate:.3f}",
})

**The observed target**

| Measure | Value |
| --- | --- |
| pages with a persistent shortfall | 1,632 of 25,214 |
| base rate (chance of a random pick being right) | 6.5% |

**CTR-gap ranking, judged two ways (Precision@50)**

| Measure | Value |
| --- | ---: |
| against its own self-defined label | 0.46 |
| against the observed window-B target | 0.04 |
| base rate, for comparison | 0.065 |

**Finding:** 6.5% of pages show a persistent shortfall, so random picking would be right about 6.5% of the time. Judged against a label made from its own gap, the CTR-gap ranking scores 0.46. Judged against what actually happened the following month, it scores 0.04, _below_ random: its top picks are small pages whose shortfall was luck and did not repeat. The same ranking looks more than ten times better when it grades its own homework, which is why every evaluation in this project uses an outcome from a window the score never saw.


## 3. Success metric

**Primary: Precision@50 against `persistent_shortfall`**, always printed beside the base rate. Of the 50 pages an editor would open, how many showed a real and substantial shortfall the following month?

**Secondary:**

- **Precision@20**, the view for a smaller review cycle.
- **Capture@50**, the share of all window-B missed clicks held by the top 50 pages. Precision counts pages; capture counts the clicks those pages represent, which is what the business actually loses.

**Locked before any model is trained:**

- A method only counts as useful if its Precision@50 is at least **3 times the base rate**.
- A learned model replaces the transparent baseline only if it beats the baseline's Precision@50 **without lowering Capture@50**, on a client holdout. Confidence intervals will then decide whether a gap between methods is real or noise.

**Method:** this cell scores four simple ranking methods on the same pages and the same target, so the bar any later model must clear is on the record before modelling starts. The last row is this project's transparent baseline.


In [5]:
# Four transparent ranking methods, all scored on window A and judged on window B.
value = lane["missed_B"]
methods = {
    "Raw CTR, lowest first": -lane["ctr_A"],
    "CTR gap vs tier median": gap_score,
    "Missed clicks, tier expectation": (lane["expected_A_tier"] - lane["clicks_prev_30d"]).clip(lower=0),
    "Missed clicks from real shortfalls (baseline)": lane["score_A"],
}
results = pd.DataFrame([
    {
        "Method": name,
        "Precision@20": f"{precision_at_k(s, target, 20):.2f}",
        "Precision@50": f"{precision_at_k(s, target, 50):.2f}",
        "Lift @50 (x random)": f"{precision_at_k(s, target, 50) / base_rate:.1f}",
        "Capture@50": f"{capture_at_k(s, value, 50):.1%}",
    }
    for name, s in methods.items()
]).set_index("Method")
show_table("Four ranking methods compared on the observed target", results)
show("The bar a learned model must clear", {
    "base rate": f"{base_rate:.3f}",
    "minimum useful Precision@50 (3x base rate)": f"{3 * base_rate:.3f}",
    "a model must beat": "the baseline row, on a client holdout, without lowering Capture@50",
})

**Four ranking methods compared on the observed target**

| Method | Precision@20 | Precision@50 | Lift @50 (x random) | Capture@50 |
| --- | ---: | ---: | ---: | ---: |
| Raw CTR, lowest first | 0.00 | 0.02 | 0.3 | 0.0% |
| CTR gap vs tier median | 0.00 | 0.04 | 0.6 | 0.1% |
| Missed clicks, tier expectation | 0.75 | 0.76 | 11.7 | 9.5% |
| Missed clicks from real shortfalls (baseline) | 0.85 | 0.84 | 13.0 | 10.9% |

**The bar a learned model must clear**

| Measure | Value |
| --- | --- |
| base rate | 0.065 |
| minimum useful Precision@50 (3x base rate) | 0.194 |
| a model must beat | the baseline row, on a client holdout, without lowering Capture@50 |

**Finding:** the two CTR-based methods do worse than random (lift below 1). Counting missed clicks instead lifts precision to 0.76, and adjusting for the client and keeping only real shortfalls lifts it to 0.84, about 13 times better than random. The baseline already clears the bar by a wide margin, which sets an honest challenge: a learned model has to beat a strong, explainable rule, and if it cannot, the rule is the better tool and that is the result I will report.


## 4. The unit of analysis, as a real dataframe

**One row = one content page** (`content_id`), carrying 90-day totals plus two separate 30-day windows.

**Method:** this cell shows how the 30,000 starter rows narrow down to the lane's pages, labels every column by its role (and why), checks that each page appears exactly once, and displays the eight highest-scoring pages with their window-B outcome.


In [6]:
# How the starter rows narrow to the lane's pages.
funnel = pd.DataFrame({
    "Pages": [f"{len(df):,}", f"{(df['avg_position'] > 0).sum():,}", f"{len(lane):,}"],
}, index=pd.Index(["All starter rows", "With real position data", "With impressions in both 30-day windows"], name="Step"))
show_table("How the starter rows narrow to the lane's pages", funnel)

# What each column is for, and why.
roles = pd.DataFrame([
    ("content_id, client_id", "Grouping and splitting only", "Pseudonyms; never a model input"),
    ("position_tier, avg_position", "Input to expected clicks", "90-day average, so it overlaps window B"),
    ("impressions_prev_30d, clicks_prev_30d", "Score input (window A)", "The only clicks the score may see"),
    ("impressions_last_30d, clicks_last_30d", "Target only (window B)", "Never enters the score"),
    ("sessions_90d, engagement_rate", "Reason code", "Changes the action; only where analytics data exists"),
    ("trend_direction, trend_pct", "Excluded", "The starter's decline label is built from these"),
], columns=["Columns", "Role", "Why"]).set_index("Columns")
show_table("What each column is used for", roles)

assert lane["content_id"].is_unique, "Grain broken: a page appears more than once."
show("Grain check", {"each row is one unique page": "passed"})

top8 = lane.sort_values("score_A", ascending=False).head(8)
show_table("The eight highest-scoring pages and what happened next", pd.DataFrame({
    "Tier": top8["position_tier"],
    "Impressions (A)": top8["impressions_prev_30d"].map("{:,}".format),
    "Clicks (A)": top8["clicks_prev_30d"].map("{:,}".format),
    "Expected (A)": top8["expected_A"].map("{:,.0f}".format),
    "Score": top8["score_A"].map("{:,.0f}".format),
    "Clicks (B)": top8["clicks_last_30d"].map("{:,}".format),
    "Persistent shortfall (B)": top8["persistent_shortfall"].map({1: "yes", 0: "no"}),
}).set_index(top8["content_id"]).rename_axis("Page (pseudonym)"))

**How the starter rows narrow to the lane's pages**

| Step | Pages |
| --- | ---: |
| All starter rows | 30,000 |
| With real position data | 28,795 |
| With impressions in both 30-day windows | 25,214 |

**What each column is used for**

| Columns | Role | Why |
| --- | --- | --- |
| content_id, client_id | Grouping and splitting only | Pseudonyms; never a model input |
| position_tier, avg_position | Input to expected clicks | 90-day average, so it overlaps window B |
| impressions_prev_30d, clicks_prev_30d | Score input (window A) | The only clicks the score may see |
| impressions_last_30d, clicks_last_30d | Target only (window B) | Never enters the score |
| sessions_90d, engagement_rate | Reason code | Changes the action; only where analytics data exists |
| trend_direction, trend_pct | Excluded | The starter's decline label is built from these |

**Grain check**

| Measure | Value |
| --- | --- |
| each row is one unique page | passed |

**The eight highest-scoring pages and what happened next**

| Page (pseudonym) | Tier | Impressions (A) | Clicks (A) | Expected (A) | Score | Clicks (B) | Persistent shortfall (B) |
| --- | --- | ---: | ---: | ---: | ---: | ---: | --- |
| content_8c19996aa890 | top_3 | 161,284 | 389 | 822 | 433 | 251 | no |
| content_5fe46e04994d | page_1 | 218,786 | 250 | 676 | 426 | 220 | no |
| content_c8e9d6ab9013 | page_1 | 111,885 | 0 | 317 | 317 | 0 | yes |
| content_36ff89c8214e | page_1 | 106,412 | 63 | 301 | 238 | 35 | yes |
| content_1276a0c674ce | page_1 | 46,183 | 74 | 271 | 197 | 30 | yes |
| content_0bebf38cb9b3 | page_1 | 33,618 | 42 | 232 | 190 | 50 | yes |
| content_91652435f57a | page_1 | 75,160 | 31 | 213 | 182 | 49 | yes |
| content_cb112fce36be | page_1 | 124,500 | 171 | 352 | 181 | 134 | no |

## 5. Why ML beats a fixed rule here

A fixed rule needs one expected rate per position tier and one minimum-traffic cutoff. Three tests show where the data breaks that: the client a page belongs to, its content type, and its amount of traffic.


### 5a. The client matters, not only the tier

**Method:** this cell compares clients' overall click rates, shows how concentrated the CTR-gap rule's flags are among clients, and checks whether adjusting the expectation for the client improves Precision@50 against the observed target.


In [7]:
# Clients' overall click rates, and how much of the CTR-gap rule's output comes from only three of them.
pos = df[df["avg_position"] > 0]
client = pos.groupby("client_id").agg(clicks=("clicks_90d", "sum"), impressions=("impressions_90d", "sum"))
client_ctr = 100 * client["clicks"] / client["impressions"]
q = client_ctr.quantile([0.25, 0.75])
top3_flag_share = lane.loc[gap_flag, "client_id"].value_counts().head(3).sum() / gap_flag.sum()
show("How much clients differ", {
    "CTR of the middle half of clients": f"{q[0.25]:.2f}% to {q[0.75]:.2f}%",
    "CTR of the highest client": f"{client_ctr.max():.2f}%",
    "CTR-gap flags coming from just 3 clients": f"{top3_flag_share:.0%}",
})

# Does adjusting the expectation for the client improve the ranking? Judged on the window-B target.
tier_only = (lane["expected_A_tier"] - lane["clicks_prev_30d"]).clip(lower=0)
show_table("Precision@50 with and without the client adjustment", pd.DataFrame({
    "Expected clicks based on": ["Position tier only", "Position tier and client"],
    "Precision@50": [f"{precision_at_k(tier_only, target, 50):.2f}", f"{precision_at_k(lane['missed_A'], target, 50):.2f}"],
}).set_index("Expected clicks based on"))

**How much clients differ**

| Measure | Value |
| --- | --- |
| CTR of the middle half of clients | 0.23% to 0.45% |
| CTR of the highest client | 3.04% |
| CTR-gap flags coming from just 3 clients | 58% |

**Precision@50 with and without the client adjustment**

| Expected clicks based on | Precision@50 |
| --- | ---: |
| Position tier only | 0.76 |
| Position tier and client | 0.84 |

**Finding:** clients' overall CTR ranges widely (the middle half sits between 0.23% and 0.45%, and the highest client reaches 3.04%), and 58% of the CTR-gap flags come from just three clients. A tier-only expectation hands a low-CTR client's whole site to the editor as "opportunities". A simple client adjustment already lifts Precision@50 from 0.76 to 0.84.


### 5b. Content type moves the expected rate inside one tier

**Method:** this cell takes only the page-1 pages and computes the pooled CTR separately for each content type.


In [8]:
# Within one tier (page 1), do content types click at the same rate?
page1 = pos[pos["position_tier"] == "page_1"]
by_type = 100 * page1.groupby("content_type")["clicks_90d"].sum() / page1.groupby("content_type")["impressions_90d"].sum()
show_table("Pooled CTR on page 1, by content type", pd.DataFrame({
    "Pages": page1.groupby("content_type").size().map("{:,}".format),
    "Pooled CTR": by_type.map("{:.2f}%".format),
}).rename_axis("Content type"))

**Pooled CTR on page 1, by content type**

| Content type | Pages | Pooled CTR |
| --- | ---: | ---: |
| comparison article | 435 | 0.08% |
| feedly article | 878 | 0.61% |
| keyword article | 10,501 | 0.35% |

**Finding:** within page 1 alone, comparison articles earn 0.08%, keyword articles 0.35%, and feedly articles 0.61%. One rate per tier mislabels whole content types.


### 5c. One impression cutoff cannot separate real gaps from luck

**Method:** this cell first works out how many impressions a page in each tier needs before it is expected to earn 5 clicks, which is roughly where a shortfall starts to mean something. It then ranks pages by CTR gap behind three different impression cutoffs and compares each with the missed-clicks score, which uses no cutoff.


In [9]:
# Impressions a page needs before it is expected to earn 5 clicks; the number depends on the tier.
rate = pos.groupby("position_tier")["clicks_90d"].sum() / pos.groupby("position_tier")["impressions_90d"].sum()
show_table("Impressions needed to expect 5 clicks, by position tier", pd.DataFrame({
    "Impressions needed": (5 / rate).reindex(TIERS).round(-1).astype(int).map("{:,}".format),
}).rename_axis("Position tier"))

# Precision@50 of the CTR-gap ranking behind different impression cutoffs, against the cutoff-free score.
cutoffs = {}
for floor in (100, 1000, 5000):
    s = lane[lane["impressions_prev_30d"] >= floor]
    gap = s.groupby("position_tier")["ctr_A"].transform("median") - s["ctr_A"]
    cutoffs[f"CTR-gap ranking, impressions of {floor:,} or more"] = f"{precision_at_k(gap, target, 50):.2f}"
cutoffs["Missed clicks from real shortfalls, no cutoff"] = f"{precision_at_k(lane['score_A'], target, 50):.2f}"
show_table("Precision@50 behind an impression cutoff", pd.DataFrame({
    "Precision@50": list(cutoffs.values()),
}, index=pd.Index(list(cutoffs), name="Ranking")))

**Impressions needed to expect 5 clicks, by position tier**

| Position tier | Impressions needed |
| --- | ---: |
| top_3 | 1,020 |
| page_1 | 1,430 |
| striking | 1,440 |
| page_3_5 | 3,230 |
| deep | 12,090 |

**Precision@50 behind an impression cutoff**

| Ranking | Precision@50 |
| --- | ---: |
| CTR-gap ranking, impressions of 100 or more | 0.06 |
| CTR-gap ranking, impressions of 1,000 or more | 0.20 |
| CTR-gap ranking, impressions of 5,000 or more | 0.56 |
| Missed clicks from real shortfalls, no cutoff | 0.84 |

**Finding:** a top-3 page is expected to earn 5 clicks after about 1,000 impressions; a deep page needs about 12,000. Any single impression cutoff is too strict for one tier and too loose for another. Even a well-chosen cutoff does not fix the ranking: the CTR-gap ranking reaches 0.56 behind a 5,000-impression cutoff, still well short of the 0.84 the missed-clicks score reaches with no cutoff at all, because sorting by gap ignores how many clicks are at stake.


### Where ML earns its place, and where it may not

The strongest use of a model here is a better _expected-clicks_ estimate that combines precise position, client, content type, and intent, and that still works on clients it has never seen. The score on top stays transparent. Because the baseline is already strong, a learned expectation has to beat it on clients it has never seen; if it cannot, the baseline is the better tool, and that is a result too.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`.
